# ⚡ Project 8 — TriggerMind: Event-Triggered Automation Agent

**Core Concept:** React to events automatically with retry logic and dead letter handling

### Architecture
Event → Queue → Classifier → Handler → Action → Log
                                    ↓
                              Retry on Failure
                                    ↓
                            Dead Letter Queue

Install

In [1]:
!pip install -q langchain langchain-groq langchain-core loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.1 MB/s eta 0:00:00


API Key

In [2]:
import os
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

All Setup In One Block

In [3]:
import os
import time
import json
import uuid
from datetime import datetime, timezone
from enum import Enum
from loguru import logger
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import sys

logger.remove()
logger.add(sys.stdout, format="{time:HH:mm:ss} | {level} | {message}", level="DEBUG")

# ── Event Types ──────────────────────────────────────────────
class EventType(Enum):
    GITHUB_PUSH = "github_push"
    NEW_ORDER = "new_order"
    PRICE_CHANGE = "price_change"
    USER_SIGNUP = "user_signup"
    PAYMENT_FAILED = "payment_failed"
    SERVER_ALERT = "server_alert"
    SUPPORT_TICKET = "support_ticket"

class EventStatus(Enum):
    PENDING = "pending"
    PROCESSING = "processing"
    COMPLETED = "completed"
    FAILED = "failed"
    DEAD_LETTER = "dead_letter"

# ── Event Model ──────────────────────────────────────────────
class Event:
    def __init__(self, event_type: EventType, payload: dict,
                 max_retries: int = 3):
        self.id = str(uuid.uuid4())[:8]
        self.event_type = event_type
        self.payload = payload
        self.status = EventStatus.PENDING
        self.created_at = datetime.now(timezone.utc).isoformat()
        self.processed_at = None
        self.retry_count = 0
        self.max_retries = max_retries
        self.result = None
        self.error = None

    def to_dict(self) -> dict:
        return {
            "id": self.id,
            "event_type": self.event_type.value,
            "payload": self.payload,
            "status": self.status.value,
            "created_at": self.created_at,
            "processed_at": self.processed_at,
            "retry_count": self.retry_count,
            "result": self.result,
            "error": self.error
        }

# ── Event Handlers ───────────────────────────────────────────
class EventHandlers:
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are TriggerMind, an event processing agent.
Process the given event and provide a clear action response.
Be specific and actionable. Keep response under 100 words."""),
            ("human", "Event Type: {event_type}\nPayload: {payload}\n\nWhat action should be taken?")
        ])
        self.chain = self.prompt | self.llm

    def process_github_push(self, payload: dict) -> str:
        response = self.chain.invoke({
            "event_type": "GitHub Push",
            "payload": json.dumps(payload)
        })
        return f"Code Review Triggered: {response.content[:200]}"

    def process_new_order(self, payload: dict) -> str:
        order_id = payload.get("order_id", "unknown")
        amount = payload.get("amount", 0)
        response = self.chain.invoke({
            "event_type": "New Order",
            "payload": json.dumps(payload)
        })
        return f"Order {order_id} (${amount}) processed: {response.content[:150]}"

    def process_price_change(self, payload: dict) -> str:
        product = payload.get("product", "unknown")
        old_price = payload.get("old_price", 0)
        new_price = payload.get("new_price", 0)
        change = ((new_price - old_price) / old_price * 100) if old_price > 0 else 0
        return f"Price alert: {product} changed by {change:.1f}% (${old_price} → ${new_price}). Notifications sent."

    def process_user_signup(self, payload: dict) -> str:
        user = payload.get("username", "unknown")
        email = payload.get("email", "unknown")
        response = self.chain.invoke({
            "event_type": "User Signup",
            "payload": json.dumps(payload)
        })
        return f"Welcome flow triggered for {user} ({email}): {response.content[:150]}"

    def process_payment_failed(self, payload: dict) -> str:
        user = payload.get("user_id", "unknown")
        amount = payload.get("amount", 0)
        response = self.chain.invoke({
            "event_type": "Payment Failed",
            "payload": json.dumps(payload)
        })
        return f"Payment recovery initiated for user {user} (${amount}): {response.content[:150]}"

    def process_server_alert(self, payload: dict) -> str:
        severity = payload.get("severity", "unknown")
        service = payload.get("service", "unknown")
        if severity == "critical":
            return f"CRITICAL ALERT: {service} is down. PagerDuty notified. On-call engineer alerted. Incident #INC-{uuid.uuid4().hex[:6].upper()} created."
        return f"Server alert for {service} (severity: {severity}). Monitoring team notified."

    def process_support_ticket(self, payload: dict) -> str:
        ticket_id = payload.get("ticket_id", "unknown")
        priority = payload.get("priority", "normal")
        response = self.chain.invoke({
            "event_type": "Support Ticket",
            "payload": json.dumps(payload)
        })
        return f"Ticket {ticket_id} (priority: {priority}) routed: {response.content[:150]}"

# ── Event Queue ──────────────────────────────────────────────
class EventQueue:
    def __init__(self):
        self.queue = []
        self.dead_letter_queue = []
        self.processed = []

    def enqueue(self, event: Event):
        self.queue.append(event)
        logger.info(f"Event enqueued: {event.event_type.value} | ID: {event.id}")

    def dequeue(self) -> Event:
        if self.queue:
            return self.queue.pop(0)
        return None

    def send_to_dead_letter(self, event: Event):
        event.status = EventStatus.DEAD_LETTER
        self.dead_letter_queue.append(event)
        logger.error(f"Event sent to dead letter queue: {event.id}")

    def get_stats(self) -> dict:
        return {
            "pending": len(self.queue),
            "processed": len(self.processed),
            "dead_letter": len(self.dead_letter_queue),
            "total": len(self.queue) + len(self.processed) + len(self.dead_letter_queue)
        }

# ── TriggerMind Agent ────────────────────────────────────────
class TriggerMindAgent:
    def __init__(self):
        self.llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0.1,
            api_key=os.environ["GROQ_API_KEY"]
        )
        self.queue = EventQueue()
        self.handlers = EventHandlers(self.llm)
        self.handler_map = {
            EventType.GITHUB_PUSH: self.handlers.process_github_push,
            EventType.NEW_ORDER: self.handlers.process_new_order,
            EventType.PRICE_CHANGE: self.handlers.process_price_change,
            EventType.USER_SIGNUP: self.handlers.process_user_signup,
            EventType.PAYMENT_FAILED: self.handlers.process_payment_failed,
            EventType.SERVER_ALERT: self.handlers.process_server_alert,
            EventType.SUPPORT_TICKET: self.handlers.process_support_ticket
        }
        logger.info("TriggerMind agent initialized")

    def trigger_event(self, event_type: EventType, payload: dict) -> Event:
        event = Event(event_type, payload)
        self.queue.enqueue(event)
        return event

    def process_event(self, event: Event) -> dict:
        logger.info(f"Processing event: {event.event_type.value} | ID: {event.id}")
        event.status = EventStatus.PROCESSING

        while event.retry_count <= event.max_retries:
            try:
                handler = self.handler_map.get(event.event_type)
                if not handler:
                    raise ValueError(f"No handler for {event.event_type.value}")

                result = handler(event.payload)
                event.status = EventStatus.COMPLETED
                event.result = result
                event.processed_at = datetime.now(timezone.utc).isoformat()
                self.queue.processed.append(event)
                logger.info(f"Event completed: {event.id} | Attempts: {event.retry_count + 1}")
                return event.to_dict()

            except Exception as e:
                event.retry_count += 1
                event.error = str(e)
                logger.warning(f"Event failed (attempt {event.retry_count}): {event.id} | Error: {e}")

                if event.retry_count > event.max_retries:
                    self.queue.send_to_dead_letter(event)
                    return event.to_dict()

                time.sleep(0.5)

        return event.to_dict()

    def process_all(self) -> list:
        results = []
        while self.queue.queue:
            event = self.queue.dequeue()
            result = self.process_event(event)
            results.append(result)
        return results

    def display_result(self, result: dict):
        print("\n" + "="*55)
        print("TRIGGERMIND EVENT RESULT")
        print("="*55)
        print(f"Event ID    : {result['id']}")
        print(f"Event Type  : {result['event_type']}")
        print(f"Status      : {result['status'].upper()}")
        print(f"Retry Count : {result['retry_count']}")
        print(f"Result      : {str(result.get('result', result.get('error', 'N/A')))[:200]}")
        print("="*55)

agent = TriggerMindAgent()
print("TriggerMind agent ready")

04:44:37 | INFO | TriggerMind agent initialized
TriggerMind agent ready


Trigger GitHub Events

In [9]:
print("========== GITHUB EVENTS ==========\n")

agent.trigger_event(EventType.GITHUB_PUSH, {
    "repo": "agentic-ai-portfolio",
    "branch": "main",
    "author": "murali",
    "commit_message": "Add memory mesh agent",
    "files_changed": ["agent.py", "memory.py"]
})

results = agent.process_all()
for result in results:
    agent.display_result(result)

========== GITHUB EVENTS ==========

04:44:54 | INFO | Event enqueued: github_push | ID: eaaef258
04:44:54 | INFO | Processing event: github_push | ID: eaaef258
04:44:54 | INFO | Event completed: eaaef258 | Attempts: 1

TRIGGERMIND EVENT RESULT
Event ID    : eaaef258
Event Type  : github_push
Status      : COMPLETED
Retry Count : 0
Result      : Code Review Triggered: Trigger a CI/CD pipeline build for the "agentic-ai-portfolio" repository, targeting the "main" branch, to validate the changes in "agent.py" and "memory.py" and ensure the "Add 


Trigger Business Events

In [5]:
print("========== BUSINESS EVENTS ==========\n")

agent.trigger_event(EventType.NEW_ORDER, {
    "order_id": "ORD-2026-001",
    "customer": "Murali Krishna",
    "amount": 299.99,
    "items": ["Python Course", "AI Bootcamp"]
})

agent.trigger_event(EventType.USER_SIGNUP, {
    "username": "murali_ai",
    "email": "muralikrishnak134@gmail.com",
    "plan": "pro",
    "source": "linkedin"
})

agent.trigger_event(EventType.PAYMENT_FAILED, {
    "user_id": "USR-4521",
    "amount": 99.99,
    "reason": "insufficient_funds",
    "attempt": 1
})

results = agent.process_all()
for result in results:
    agent.display_result(result)

========== BUSINESS EVENTS ==========

04:44:38 | INFO | Event enqueued: new_order | ID: c21e3709
04:44:38 | INFO | Event enqueued: user_signup | ID: 1240d7c6
04:44:38 | INFO | Event enqueued: payment_failed | ID: 5ed89965
04:44:38 | INFO | Processing event: new_order | ID: c21e3709
04:44:38 | INFO | Event completed: c21e3709 | Attempts: 1
04:44:38 | INFO | Processing event: user_signup | ID: 1240d7c6
04:44:39 | INFO | Event completed: 1240d7c6 | Attempts: 1
04:44:39 | INFO | Processing event: payment_failed | ID: 5ed89965
04:44:39 | INFO | Event completed: 5ed89965 | Attempts: 1

TRIGGERMIND EVENT RESULT
Event ID    : c21e3709
Event Type  : new_order
Status      : COMPLETED
Retry Count : 0
Result      : Order ORD-2026-001 ($299.99) processed: Send confirmation email to Murali Krishna with order details (ORD-2026-001, $299.99, items: Python Course, AI Bootcamp). Update order status to "Pendi

TRIGGERMIND EVENT RESULT
Event ID    : 1240d7c6
Event Type  : user_signup
Status      : COMPLE

Trigger Alert Events

In [6]:
print("========== ALERT EVENTS ==========\n")

agent.trigger_event(EventType.SERVER_ALERT, {
    "service": "payment-service",
    "severity": "critical",
    "error": "Connection timeout",
    "affected_users": 1250
})

agent.trigger_event(EventType.PRICE_CHANGE, {
    "product": "AI Pro Plan",
    "old_price": 99.99,
    "new_price": 79.99,
    "reason": "promotional_discount"
})

agent.trigger_event(EventType.SUPPORT_TICKET, {
    "ticket_id": "TKT-8821",
    "user": "customer@example.com",
    "priority": "high",
    "issue": "Cannot access premium features after payment"
})

results = agent.process_all()
for result in results:
    agent.display_result(result)

========== ALERT EVENTS ==========

04:44:39 | INFO | Event enqueued: server_alert | ID: 9cd299c0
04:44:39 | INFO | Event enqueued: price_change | ID: 748e546b
04:44:39 | INFO | Event enqueued: support_ticket | ID: 616e190c
04:44:39 | INFO | Processing event: server_alert | ID: 9cd299c0
04:44:39 | INFO | Event completed: 9cd299c0 | Attempts: 1
04:44:39 | INFO | Processing event: price_change | ID: 748e546b
04:44:39 | INFO | Event completed: 748e546b | Attempts: 1
04:44:39 | INFO | Processing event: support_ticket | ID: 616e190c
04:44:40 | INFO | Event completed: 616e190c | Attempts: 1

TRIGGERMIND EVENT RESULT
Event ID    : 9cd299c0
Event Type  : server_alert
Status      : COMPLETED
Retry Count : 0
Result      : CRITICAL ALERT: payment-service is down. PagerDuty notified. On-call engineer alerted. Incident #INC-3D9425 created.

TRIGGERMIND EVENT RESULT
Event ID    : 748e546b
Event Type  : price_change
Status      : COMPLETED
Retry Count : 0
Result      : Price alert: AI Pro Plan change

Queue Stats

In [7]:
print("========== QUEUE STATISTICS ==========\n")
stats = agent.queue.get_stats()
print(f"Pending      : {stats['pending']}")
print(f"Processed    : {stats['processed']}")
print(f"Dead Letter  : {stats['dead_letter']}")
print(f"Total Events : {stats['total']}")

print(f"\nProcessed Events:")
for event in agent.queue.processed:
    print(f"  [{event.id}] {event.event_type.value:20} | {event.status.value:10} | Retries: {event.retry_count}")

========== QUEUE STATISTICS ==========

Pending      : 0
Processed    : 7
Dead Letter  : 0
Total Events : 7

Processed Events:
  [d22481da] github_push          | completed  | Retries: 0
  [c21e3709] new_order            | completed  | Retries: 0
  [1240d7c6] user_signup          | completed  | Retries: 0
  [5ed89965] payment_failed       | completed  | Retries: 0
  [9cd299c0] server_alert         | completed  | Retries: 0
  [748e546b] price_change         | completed  | Retries: 0
  [616e190c] support_ticket       | completed  | Retries: 0


Project Summary

In [8]:
print("========== TRIGGERMIND SUMMARY ==========\n")
print("Project      : TriggerMind — Event-Triggered Automation Agent")
print("Author       : K Murali Krishna")
print("Model        : Groq LLaMA-3.3-70b-versatile")
print("\nSupported Event Types:")
for event_type in EventType:
    print(f"  ✓ {event_type.value}")
print("\nKey Capabilities:")
print("  ✓ Event queue with FIFO processing")
print("  ✓ Automatic retry with max retry limit")
print("  ✓ Dead letter queue for failed events")
print("  ✓ Event-specific handlers")
print("  ✓ Full audit trail per event")
print("\nProduction Concepts Demonstrated:")
print("  ✓ Event-driven architecture")
print("  ✓ Queue-based processing")
print("  ✓ Retry logic with backoff")
print("  ✓ Dead letter queue pattern")
print("  ✓ Idempotent event handling")

========== TRIGGERMIND SUMMARY ==========

Project      : TriggerMind — Event-Triggered Automation Agent
Author       : K Murali Krishna
Model        : Groq LLaMA-3.3-70b-versatile

Supported Event Types:
  ✓ github_push
  ✓ new_order
  ✓ price_change
  ✓ user_signup
  ✓ payment_failed
  ✓ server_alert
  ✓ support_ticket

Key Capabilities:
  ✓ Event queue with FIFO processing
  ✓ Automatic retry with max retry limit
  ✓ Dead letter queue for failed events
  ✓ Event-specific handlers
  ✓ Full audit trail per event

Production Concepts Demonstrated:
  ✓ Event-driven architecture
  ✓ Queue-based processing
  ✓ Retry logic with backoff
  ✓ Dead letter queue pattern
  ✓ Idempotent event handling
